# Module 4: Search in Azure DocumentDB

**Time**: ~75 min  
**Environment**: Jupyter notebook in VS Code

Before starting, open a PowerShell terminal in this notebook's folder and run `az login`, followed by `../../../1-DocumentDB-Introduction-and-Cluster-Setup/Set-LabEnvironment.ps1`. Select the workshop resources when prompted, then restart the notebook kernel and run each cell in order.

This completed reference uses Microsoft Entra ID for Azure DocumentDB and Azure OpenAI. It creates embeddings, stores sample documents, creates available search indexes, and demonstrates vector, BM25, fuzzy, phrase, and hybrid search.

> Full-text search is a gated preview. When it is unavailable, the notebook reports code 115 and continues with vector search. DiskANN requires an M30 or higher cluster tier.


## Step 0: Connect and configure embeddings

The required Python packages are provided on the workshop VM. This cell reads the three values produced by the shared setup script. Both clients authenticate through `AzureCliCredential`; no access keys are required.


In [ ]:
import os

from azure.identity import AzureCliCredential, get_bearer_token_provider
from pymongo import MongoClient
from pymongo.auth_oidc import OIDCCallback, OIDCCallbackContext, OIDCCallbackResult
from pymongo.errors import OperationFailure
from openai import OpenAI

class AzureIdentityTokenCallback(OIDCCallback):
    def __init__(self, credential):
        self.credential = credential

    def fetch(self, context: OIDCCallbackContext) -> OIDCCallbackResult:
        del context
        token = self.credential.get_token("https://ossrdbms-aad.database.windows.net/.default")
        return OIDCCallbackResult(access_token=token.token)

cluster_name = os.environ["DOCUMENTDB_CLUSTER_NAME"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
embedding_model = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
credential = AzureCliCredential()
client = MongoClient(
    f"mongodb+srv://{cluster_name}.global.mongocluster.cosmos.azure.com/",
    tls=True,
    retryWrites=False,
    authMechanism="MONGODB-OIDC",
    authMechanismProperties={"OIDC_CALLBACK": AzureIdentityTokenCallback(credential)},
)
db = client["docdbworkshop"]
collection = db["workshop_content"]
openai_client = OpenAI(
    base_url=f"{azure_openai_endpoint.rstrip('/')}/openai/v1/",
    api_key=get_bearer_token_provider(credential, "https://ai.azure.com/.default"),
)
print(db.command({"ping": 1}))
print("Embedding deployment:", embedding_model)


## Step 1: Generate embeddings and load sample documents

This cell calls the OpenAI embeddings API for each sample document body, stores the resulting vector in the `embedding` field, and inserts the documents into Azure DocumentDB.

In [ ]:
def create_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(model=embedding_model, input=text)
    return response.data[0].embedding

source_docs = [
    {"_id":"doc-search-001","title":"DiskANN vector indexing","category":"vector","body":"Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents.","sku":"SEARCH-VEC-001"},
    {"_id":"doc-search-002","title":"BM25 keyword search","category":"full-text","body":"Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata.","sku":"SEARCH-FTS-001"},
    {"_id":"doc-search-003","title":"Hybrid search with RRF","category":"hybrid","body":"Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion.","sku":"SEARCH-HYB-001"},
    {"_id":"doc-search-004","title":"RAG grounding","category":"rag","body":"Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context.","sku":"RAG-PIPE-001"},
    {"_id":"doc-search-005","title":"Operational filtering","category":"filters","body":"Search applications often filter by status, tenant, region, stock, or category after the search stage narrows candidate documents.","sku":"SEARCH-FLT-001"}
]

collection.drop()
for doc in source_docs:
    doc["embedding"] = create_embedding(doc["body"])
collection.insert_many(source_docs)
embedding_dimensions = len(source_docs[0]["embedding"])
print("Loaded documents:", collection.count_documents({}))
print("Embedding dimensions:", embedding_dimensions)

## Step 2: Create vector and full-text indexes

The vector index uses `cosmosSearch` and the actual embedding dimension returned by OpenAI. The full-text index uses `createSearchIndexes` over the `body` field.

In [ ]:
vector_index_result = db.command({
    "createIndexes": "workshop_content",
    "indexes": [{
        "name": "idx_embedding_diskann",
        "key": {"embedding": "cosmosSearch"},
        "cosmosSearchOptions": {"kind": "vector-diskann", "dimensions": embedding_dimensions, "similarity": "COS", "maxDegree": 32, "lBuild": 64}
    }]
})

full_text_search_supported = True
try:
    full_text_index_result = db.command({
        "createSearchIndexes": "workshop_content",
        "indexes": [{
            "name": "idx_body_fts",
            "definition": {"mappings": {"dynamic": False, "fields": {"body": {"type": "string"}}}}
        }]
    })
except OperationFailure as error:
    if error.code != 115:
        raise
    full_text_search_supported = False
    full_text_index_result = {"ok": 0, "code": error.code, "message": "Full-text search is not enabled; continue with vector search."}

{"vector": vector_index_result, "fullText": full_text_index_result}


## Step 3: Generate a query embedding and run vector search

The query text is embedded with the same model, then used as the `vector` in `$search.cosmosSearch`.

In [ ]:
search_text = "semantic retrieval for RAG"
query_vector = create_embedding(search_text)
vector_results = list(collection.aggregate([
    {"$search": {"cosmosSearch": {"path": "embedding", "vector": query_vector, "k": 3}}},
    {"$project": {"_id": 0, "title": 1, "category": 1, "body": 1, "score": {"$meta": "searchScore"}}}
]))
vector_results

## Step 4: Run BM25 full-text search

This query uses `$search.text` against the named full-text index and projects BM25 relevance scores with `$meta: "searchScore"`.

In [ ]:
if full_text_search_supported:
    bm25_results = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "text": {"query": "BM25 ranking", "path": "body"}}},
        {"$limit": 5},
        {"$project": {"_id": 0, "title": 1, "body": 1, "score": {"$meta": "searchScore"}}}
    ]))
else:
    bm25_results = []
    print("Skipped BM25 search because full-text search is not enabled.")
bm25_results


## Step 5: Run fuzzy and phrase search

Fuzzy search tolerates typos with `maxEdits`. Phrase search requires ordered terms, with `slop` controlling how close the words must be.

In [ ]:
if full_text_search_supported:
    fuzzy_results = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "text": {"query": "retrival augmentd genration", "path": "body", "fuzzy": {"maxEdits": 1}}}},
        {"$limit": 5},
        {"$project": {"_id": 0, "title": 1, "score": {"$meta": "searchScore"}}}
    ]))
    phrase_results = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "phrase": {"query": "Reciprocal Rank Fusion", "path": "body", "slop": 0}}},
        {"$limit": 5},
        {"$project": {"_id": 0, "title": 1, "score": {"$meta": "searchScore"}}}
    ]))
else:
    fuzzy_results, phrase_results = [], []
    print("Skipped fuzzy and phrase search because full-text search is not enabled.")
{"fuzzy": fuzzy_results, "phrase": phrase_results}


## Step 6: Run hybrid search

Hybrid search embeds the user query, runs vector and BM25 retrieval, and fuses both ranked lists with RRF.

In [ ]:
def rrf(lists, k=60, top_n=5):
    scores, docs_by_id = {}, {}
    for results in lists:
        for rank, doc in enumerate(results):
            doc_id = str(doc["_id"])
            docs_by_id[doc_id] = doc
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return [{**docs_by_id[doc_id], "rrfScore": score} for doc_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

user_query = "semantic retrieval for RAG"
query_vector = create_embedding(user_query)
vector_hits = list(collection.aggregate([
    {"$search": {"cosmosSearch": {"path": "embedding", "vector": query_vector, "k": 5}}},
    {"$project": {"_id": 1, "title": 1, "score": {"$meta": "searchScore"}}}
]))
if full_text_search_supported:
    keyword_hits = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "text": {"query": user_query, "path": "body"}}},
        {"$limit": 5},
        {"$project": {"_id": 1, "title": 1, "score": {"$meta": "searchScore"}}}
    ]))
    combined_results = rrf([keyword_hits, vector_hits])
else:
    keyword_hits = []
    combined_results = vector_hits
    print("Full-text search is unavailable; showing vector results instead of RRF.")
combined_results
